Dataset


In [8]:
import kagglehub
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'gtzan-dataset-music-genre-classification' dataset.
Path to dataset files: /kaggle/input/gtzan-dataset-music-genre-classification


In [9]:
import librosa
import numpy as np
from pathlib import Path
import random

import time
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
fft_total_time = 0  # Time to make a spectrogram for each song (10 total songs)
inference_total_time = 0
e2e_time = 0
correct_predictions = 0
total_predictions = 0
def my_spectrogram(x, m, fs):

    # Inputs:
    # x -- data vector
    # m -- block size, must be even
    # fs -- sampling rate

    if (m % 2 != 0):
        raise ValueError("Choose an even block size")

    # Pad x up to a multiple of m
    lx = len(x)
    nt = int(np.ceil(lx/m))
    # plus 1 is so that the overlapping window works
    xp = np.zeros((nt + 1) * m)
    xp[:lx] = x


    # Rectangular windowing
    # use reshape to make it an m by nt matrix
    xm = xp[:nt * m].reshape((nt, m)).T

    # 50% overlapping blocks
    n1 = m // 2
    n2 = n1+nt*m

    xmi = xp[n1:n2].reshape((nt, m)).T

    xm3 = np.zeros((m,2*nt))
    xm3[:, 0::2] = xm
    xm3[:, 1::2] = xmi

    window = np.hamming(m)
    xm3 *= window[:, None]

    xmf3 = np.fft.fft(xm3, axis=0)

    spectrogram = np.abs(xmf3[:m // 2, :])
    spectrogram = 20 * np.log10(
    np.abs(xmf3[:m // 2, :]) + 1e-10
    )
    # Axes
    freqs = np.arange(0, m // 2) * fs / m
    times = np.arange(1, 2 * nt + 1) * m / (2 * fs)

    return spectrogram, freqs, times

####################### SONG LOADING ########################



In [10]:
import soundfile as sf

def is_valid_audio(path):
    try:
        sf.info(str(path))
        return True
    except:
        return False

In [11]:
from pathlib import Path
import numpy as np
import librosa
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

songs_folder = Path(f"{path}/Data/genres_original")

chunk_size = 50
sr = 22050
fft_size = 1024

genres = sorted([d.name for d in songs_folder.iterdir() if d.is_dir()])
genre_to_label = {g: i for i, g in enumerate(genres)}

index = []

for genre in genres:
    genre_folder = songs_folder / genre
    label = genre_to_label[genre]

    count = 0
    for song_path in genre_folder.glob("*.wav"):
        # count += 1
        # if count > 5:
        #     break
        if is_valid_audio(song_path):
            index.append((song_path, label))
        else:
            print(f"Skipping corrupted file: {song_path}")



train_index, test_index = train_test_split(
    index,
    test_size=0.2,
    random_state=42,
    stratify=[y for _, y in index]
)


class AudioDataset(Dataset):
    def __init__(self, index, chunk_size, sr, fft_size, train=True, chunks_per_song=30):
        self.index = index
        self.chunk_size = chunk_size
        self.sr = sr
        self.fft_size = fft_size
        self.train = train
        self.chunks_per_song = chunks_per_song

    def __len__(self):
        return len(self.index) * self.chunks_per_song

    def __getitem__(self, idx):
        song_path, label = self.index[idx // self.chunks_per_song]

        audio, _ = librosa.load(song_path, sr=self.sr)

        spectrogram, _, _ = my_spectrogram(
            audio,
            m=self.fft_size,
            fs=self.sr
        )

        spectrogram = spectrogram.T
        spectrogram = (spectrogram - spectrogram.mean()) / (spectrogram.std() + 1e-8)

        max_start = spectrogram.shape[0] - self.chunk_size

        if self.train:
            start = np.random.randint(0, max_start)
        else:
            start = max_start // 2  # deterministic evaluation crop

        chunk = spectrogram[start:start + self.chunk_size]

        return (
            torch.tensor(chunk, dtype=torch.float32),
            torch.tensor(label, dtype=torch.long)
        )


train_dataset = AudioDataset(train_index, chunk_size, sr, fft_size, train=True)
test_dataset = AudioDataset(test_index, chunk_size, sr, fft_size, train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

Skipping corrupted file: /kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original/jazz/jazz.00054.wav


In [2]:
import torch
import torch.nn as nn

class LangIDCNNLSTM(nn.Module):
  def __init__(self):
    super().__init__()
    self.cnn = nn.Sequential(
        nn.Conv2d(in_channels=1, out_channels=16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
        nn.ReLU(),
        nn.MaxPool2d((1, 2)),
        nn.Conv2d(in_channels=16, out_channels=32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
        nn.ReLU(),
        nn.MaxPool2d((1, 2)),
    )
    self.lstm = nn.LSTM(input_size=32*128, hidden_size=128, num_layers=2, batch_first=True, dropout=0.2)
    self.fc = nn.Linear(128, 10)


  def forward(self,x):
    x = x.unsqueeze(1)
    x = self.cnn(x)
    x = x.permute(0,2,1,3)
    x = x.reshape(x.shape[0], x.shape[1], -1)
    x, _ = self.lstm(x)
    x = x[:, -1, :]
    x = self.fc(x)
    return x



In [14]:
def save_model(model, path):
    torch.save(model.state_dict(), path)

def load_model(model, path):
    model.load_state_dict(torch.load(path))
    return model

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = LangIDCNNLSTM()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
epochs = 10
model.to(device)
model.train()
for epoch in range(epochs):

    total_loss = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        predictions = model(batch_x)

        loss = criterion(predictions, batch_y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()
        save_model(model, f"model_{epoch}.pth")

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 635.0175
Epoch 2, Loss: 486.0235
Epoch 3, Loss: 375.4422
Epoch 4, Loss: 285.8811
Epoch 5, Loss: 209.6733
Epoch 6, Loss: 151.2410
Epoch 7, Loss: 113.9466
Epoch 8, Loss: 89.5613
Epoch 9, Loss: 65.3447
Epoch 10, Loss: 55.1578


In [4]:
import numpy as np

# simple test: impulse signal
frame = np.zeros(1024)
frame[0] = 1.0

np.savetxt("/Users/sarakothari/Documents/stanford/junior/spring/EE109/EE109-Final-Project/frame.csv", 
           frame, delimiter=',')
print("wrote frame.csv")

wrote frame.csv


In [ ]:
import torch
import numpy as np

def evaluate(model, loader, device="cpu"):
    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    accuracy = (all_preds == all_targets).mean() * 100

    return accuracy, all_preds, all_targets

In [ ]:

test_acc, preds, labels = evaluate(model, test_loader, device)


print("Test accuracy:", test_acc)

## Inference


In [ ]:
import torch
import numpy as np
import librosa

def my_spectrogram(x, m, fs):
    if (m % 2 != 0):
        raise ValueError("Choose an even block size")

    lx = len(x)
    nt = int(np.ceil(lx/m))
    xp = np.zeros((nt + 1) * m)
    xp[:lx] = x

    xm = xp[:nt * m].reshape((nt, m)).T

    n1 = m // 2
    n2 = n1 + nt * m
    xmi = xp[n1:n2].reshape((nt, m)).T

    xm3 = np.zeros((m, 2 * nt))
    xm3[:, 0::2] = xm
    xm3[:, 1::2] = xmi

    window = np.hamming(m)
    xm3 *= window[:, None]

    xmf3 = np.fft.fft(xm3, axis=0)

    spectrogram = 20 * np.log10(
        np.abs(xmf3[:m // 2, :]) + 1e-10
    )

    return spectrogram, None, None


class LangIDCNNLSTM(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = torch.nn.Sequential(
            torch.nn.Conv2d(1, 16, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d((1, 2)),
            torch.nn.Conv2d(16, 32, 3, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d((1, 2)),
        )
        self.lstm = torch.nn.LSTM(
            input_size=32*128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=0.2
        )
        self.fc = torch.nn.Linear(128, 10)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.cnn(x)
        x = x.permute(0, 2, 1, 3)
        x = x.reshape(x.shape[0], x.shape[1], -1)
        x, _ = self.lstm(x)
        x = x[:, -1, :]
        x = self.fc(x)
        return x


MODEL_PATH = "model_9.pth"
SR = 22050
FFT_SIZE = 1024
CHUNK_SIZE = 50

GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]

device = "cuda" if torch.cuda.is_available() else "cpu"

model = LangIDCNNLSTM()
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()


def predict_wav(file_path, num_chunks=10):
    """
    Instead of relying on a single fixed crop of the spectrogram, this function
    performs inference over multiple randomly sampled chunks from the same audio.
    During training, the model only ever sees short segments (chunks) of each song,
    not the full track. That means any single chunk at inference time may not be
    fully representative of the entire audio (e.g., intros, silence, or transitions).
    
    By sampling multiple chunks across the spectrogram and running the model on
    each one independently, we obtain several probability distributions over genres.
    These predictions are then averaged to produce a more stable and robust final
    estimate. This reduces variance and makes the prediction less sensitive to
    where the chunk was taken from, effectively approximating full-song inference
    while still using the fixed-size input the model expects.
    """

    audio, _ = librosa.load(file_path, sr=SR)

    spec, _, _ = my_spectrogram(audio, m=FFT_SIZE, fs=SR)

    spec = spec.T
    spec = (spec - spec.mean()) / (spec.std() + 1e-8)
    if spec.shape[0] < CHUNK_SIZE:
        raise ValueError("Audio too short")
    probs_list = []
    for _ in range(num_chunks):
        start = np.random.randint(0, spec.shape[0] - CHUNK_SIZE)
        chunk = spec[start:start + CHUNK_SIZE]

        x = torch.tensor(chunk, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            probs_list.append(probs)

    avg_probs = torch.mean(torch.stack(probs_list), dim=0)
    pred = torch.argmax(avg_probs, dim=1).item()

    return GENRES[pred], avg_probs.cpu().numpy()


#your file path
file_path = "test.wav"

genre, probs = predict_wav(file_path)

print(f"Predicted genre: {genre}")
print("Probabilities:", probs)

In [3]:
import torch

model = LangIDCNNLSTM()
model.load_state_dict(torch.load('model_9.pth', map_location='cpu'))

lstm = model.lstm
fc = model.fc

print("weight_ih_l0:", lstm.weight_ih_l0.shape)
print("weight_hh_l0:", lstm.weight_hh_l0.shape)
print("bias_ih_l0:",   lstm.bias_ih_l0.shape)
print("bias_hh_l0:",   lstm.bias_hh_l0.shape)
print("weight_ih_l1:", lstm.weight_ih_l1.shape)
print("weight_hh_l1:", lstm.weight_hh_l1.shape)
print("fc weight:",    fc.weight.shape)
print("fc bias:",      fc.bias.shape)

weight_ih_l0: torch.Size([512, 4096])
weight_hh_l0: torch.Size([512, 128])
bias_ih_l0: torch.Size([512])
bias_hh_l0: torch.Size([512])
weight_ih_l1: torch.Size([512, 128])
weight_hh_l1: torch.Size([512, 128])
fc weight: torch.Size([10, 128])
fc bias: torch.Size([10])
